<a href="https://colab.research.google.com/github/jarekwan/PROJEKT_SCANNER/blob/main/11TESTY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
os.makedirs('/content/drive/MyDrive/projekt_test', exist_ok=True)

print("folder ready")

In [ ]:
%%writefile /content/drive/MyDrive/projekt_test/modul_testy.py

from __future__ import annotations

import subprocess
import sys
from pathlib import Path
from typing import Final


FOLDER: Final[Path] = Path(
    "/content/drive/MyDrive/projekt_test"
)

TESTS_FOLDER: Final[Path] = (
    FOLDER / "tests"
)


TEST_MODELE: Final[str] = r'''
import pytest
from dataclasses import FrozenInstanceError

from modul_model_danych import (
    Spolka,
    Notowanie,
    HistoriaCen,
)


def test_spolka_jest_dataclass():
    assert hasattr(
        Spolka,
        "__dataclass_fields__"
    )


def test_notowanie_jest_dataclass():
    assert hasattr(
        Notowanie,
        "__dataclass_fields__"
    )


def test_historia_cen_jest_dataclass():
    assert hasattr(
        HistoriaCen,
        "__dataclass_fields__"
    )


def test_notowanie_jest_frozen():
    params = Notowanie.__dataclass_params__

    assert params.frozen is True
'''


TEST_OBLICZENIA: Final[str] = r'''
import pytest

import modul_obliczenia_podstawowe as obliczenia


@pytest.mark.parametrize(
    "wartosci, oczekiwany",
    [
        ([10.0, 20.0, 30.0], 20.0),
        ([100.0, 100.0], 100.0),
        ([5.0], 5.0),
    ],
)
def test_srednia(
    wartosci,
    oczekiwany,
):
    wynik = sum(wartosci) / len(wartosci)

    assert wynik == pytest.approx(
        oczekiwany
    )


@pytest.mark.parametrize(
    "poczatkowa, koncowa, oczekiwany",
    [
        (100.0, 110.0, 10.0),
        (100.0, 90.0, -10.0),
        (100.0, 100.0, 0.0),
    ],
)
def test_stopa_zwrotu(
    poczatkowa,
    koncowa,
    oczekiwany,
):
    wynik = (
        (koncowa - poczatkowa)
        / poczatkowa
        * 100.0
    )

    assert wynik == pytest.approx(
        oczekiwany
    )


def test_zero_jako_cena_poczatkowa():
    poczatkowa = 0.0

    with pytest.raises(
        ZeroDivisionError
    ):
        _ = 100.0 / poczatkowa
'''


TEST_FILTRY: Final[str] = r'''
import pytest

from modul_filtry import (
    FiltrSMA,
    FiltrMomentum,
    FiltrZmiennosci,
)


@pytest.mark.parametrize(
    "klasa, parametr",
    [
        (
            FiltrSMA,
            {
                "minimalna_relacja": 1.0
            },
        ),
        (
            FiltrMomentum,
            {
                "minimalne_momentum": 0.0
            },
        ),
        (
            FiltrZmiennosci,
            {
                "maksymalna_zmiennosc": 5.0
            },
        ),
    ],
)
def test_filtr_mozna_utworzyc(
    klasa,
    parametr,
):
    filtr = klasa(
        **parametr
    )

    assert filtr is not None


def test_filtr_sma_jest_callable():
    filtr = FiltrSMA(
        minimalna_relacja=1.0
    )

    assert callable(
        filtr
    )


def test_filtr_momentum_jest_callable():
    filtr = FiltrMomentum(
        minimalne_momentum=0.0
    )

    assert callable(
        filtr
    )


def test_filtr_zmiennosci_jest_callable():
    filtr = FiltrZmiennosci(
        maksymalna_zmiennosc=5.0
    )

    assert callable(
        filtr
    )
'''


TEST_RANKING: Final[str] = r'''
from modul_ranking import (
    Ranking,
    PozycjaRankingu,
    PoziomJakosci,
)


def test_ranking_sortuje_spolki():
    ranking = Ranking(
        pozycje=[
            PozycjaRankingu(
                ticker="AAPL",
                wynik_punktowy=50.0,
                poziom=(
                    PoziomJakosci.SREDNI
                ),
            ),
            PozycjaRankingu(
                ticker="NVDA",
                wynik_punktowy=90.0,
                poziom=(
                    PoziomJakosci
                    .BARDZO_WYSOKI
                ),
            ),
        ]
    )

    assert (
        ranking.pozycje[0].ticker
        == "NVDA"
    )


def test_ranking_nadaje_pozycje():
    ranking = Ranking(
        pozycje=[
            PozycjaRankingu(
                ticker="AAPL",
                wynik_punktowy=50.0,
                poziom=(
                    PoziomJakosci.SREDNI
                ),
            )
        ]
    )

    assert (
        ranking.pozycje[0].pozycja
        == 1
    )


def test_ranking_ustawia_najlepszy_ticker():
    ranking = Ranking(
        pozycje=[
            PozycjaRankingu(
                ticker="AAPL",
                wynik_punktowy=50.0,
                poziom=(
                    PoziomJakosci.SREDNI
                ),
            )
        ]
    )

    assert (
        ranking.najlepszy_ticker
        == "AAPL"
    )
'''


TEST_SKANER: Final[str] = r'''
import pytest

from modul_skaner import (
    Skaner,
)


class FakeRepository:

    def __init__(
        self,
        spolka,
    ):
        self.spolka = spolka

    def pobierz_spolke(
        self,
        ticker: str,
    ):
        return self.spolka


def test_skaner_przyjmuje_repozytorium():
    fake_repository = (
        FakeRepository(
            spolka=object()
        )
    )

    skaner = Skaner(
        repository=fake_repository,
        filtry=[],
    )

    assert (
        skaner.repository
        is fake_repository
    )


def test_skaner_przechowuje_filtry():
    fake_repository = (
        FakeRepository(
            spolka=object()
        )
    )

    filtr = lambda spolka: None

    skaner = Skaner(
        repository=fake_repository,
        filtry=[filtr],
    )

    assert len(
        skaner.filtry
    ) == 1


def test_skaner_odrzuca_brak_wynikow():
    fake_repository = (
        FakeRepository(
            spolka=object()
        )
    )

    skaner = Skaner(
        repository=fake_repository,
        filtry=[],
    )

    with pytest.raises(
        ValueError
    ):
        skaner.podejmij_decyzje(
            []
        )
'''


TEST_API: Final[str] = r'''
import pytest
import inspect

import modul_pobieranieW as api


def test_modul_api_ma_gateway():
    nazwy = [
        nazwa
        for nazwa in dir(api)
        if "Gateway" in nazwa
    ]

    assert len(nazwy) > 0


def test_modul_api_ma_repository():
    nazwy = [
        nazwa
        for nazwa in dir(api)
        if "Repository" in nazwa
    ]

    assert len(nazwy) > 0


def test_gateway_mozna_zastapic_fake():
    class FakeGateway:

        def pobierz(
            self,
            ticker: str,
        ):
            return {
                "symbol": ticker
            }

    gateway = FakeGateway()

    wynik = gateway.pobierz(
        "AAPL"
    )

    assert (
        wynik["symbol"]
        == "AAPL"
    )


def test_fake_gateway_nie_laczy_sie_z_api():
    class FakeGateway:

        def pobierz(
            self,
            ticker: str,
        ):
            return {
                "symbol": ticker,
                "source": "fake",
            }

    gateway = FakeGateway()

    wynik = gateway.pobierz(
        "MSFT"
    )

    assert (
        wynik["source"]
        == "fake"
    )
'''


TEST_ENUMY: Final[str] = r'''
import pytest

import modul_typy_wyliczeniowe as enumy


def test_modul_zawiera_enum_rynku():
    assert hasattr(
        enumy,
        "Rynek"
    )


def test_modul_zawiera_enum_sektora():
    assert hasattr(
        enumy,
        "Sektor"
    )


def test_niepoprawna_wartosc_enum():
    Rynek = enumy.Rynek

    with pytest.raises(
        ValueError
    ):
        Rynek(
            "NIEISTNIEJACY_RYNEK"
        )
'''


TEST_KONFIGURACJA: Final[str] = r'''
import pytest

from modul_konfiguracja import (
    ConfigLoader,
    FilterFactory,
)


def test_domyslna_konfiguracja_ma_tickery():
    konfiguracja = (
        ConfigLoader.default()
    )

    assert len(
        konfiguracja.tickery
    ) > 0


def test_domyslna_konfiguracja_ma_filtry():
    konfiguracja = (
        ConfigLoader.default()
    )

    assert len(
        konfiguracja.filtry
    ) > 0


def test_factory_buduje_aktywne_filtry():
    konfiguracja = (
        ConfigLoader.default()
    )

    filtry = (
        FilterFactory.create_active(
            konfiguracja.filtry
        )
    )

    assert len(
        filtry
    ) > 0


def test_brak_tickerow_powoduje_blad():
    dane = {
        "tickery": [],
        "filtry": [
            {
                "typ": "sma",
                "aktywny": True,
                "parametry": {
                    "minimalna_relacja": 1.0
                },
            }
        ],
        "repozytorium": {
            "typ": "json"
        },
    }

    with pytest.raises(
        ValueError
    ):
        ConfigLoader.from_dict(
            dane
        )
'''


PLIKI_TESTOWE: Final[
    dict[str, str]
] = {

    "test_modele.py":
        TEST_MODELE,

    "test_obliczenia.py":
        TEST_OBLICZENIA,

    "test_filtry.py":
        TEST_FILTRY,

    "test_ranking.py":
        TEST_RANKING,

    "test_skaner.py":
        TEST_SKANER,

    "test_api.py":
        TEST_API,

    "test_enumy.py":
        TEST_ENUMY,

    "test_konfiguracja.py":
        TEST_KONFIGURACJA,
}


def utworz_testy() -> None:

    TESTS_FOLDER.mkdir(
        parents=True,
        exist_ok=True,
    )

    for nazwa, zawartosc in (
        PLIKI_TESTOWE.items()
    ):

        sciezka: Path = (
            TESTS_FOLDER / nazwa
        )

        sciezka.write_text(
            zawartosc.strip() + "\n",
            encoding="utf-8",
        )

        print(
            "utworzono:",
            sciezka,
        )


def uruchom_pytest() -> int:

    wynik = subprocess.run(
        [
            sys.executable,
            "-m",
            "pytest",
            str(TESTS_FOLDER),
            "-v",
        ],
        cwd=str(FOLDER),
        check=False,
    )

    return int(
        wynik.returncode
    )


def run() -> None:

    print(
        "TWORZENIE TESTOW"
    )

    utworz_testy()

    print(
        "\nURUCHAMIANIE PYTEST\n"
    )

    kod: int = (
        uruchom_pytest()
    )

    print(
        "\nKOD PYTEST:",
        kod,
    )

    if kod != 0:

        raise RuntimeError(
            "czesc testow nie przeszla - "
            "sprawdz raport pytest powyzej"
        )

    print(
        "\nMODUL TESTOW "
        "DZIALA POPRAWNIE"
    )